# Orquestracao — carteira_auto v0.2.1

Este notebook documenta o motor de orquestracao do sistema:
**DAGEngine → Nodes → PipelineContext → Registry**

Cobre:
1. DAGEngine — registro, resolucao de dependencias, execucao
2. Node — ABC, subclasses, dependencias
3. PipelineContext — comunicacao entre nodes
4. Registry — mapeamento CLI → pipelines
5. Error handling — fail_fast vs tolerante

## 1. DAGEngine — Motor de Pipelines

O `DAGEngine` orquestra a execucao de nodes em um grafo aciclico dirigido (DAG).
Resolve dependencias via topological sort (algoritmo de Kahn).

In [ ]:
from carteira_auto.core.engine import DAGEngine, Node, PipelineContext

# Criar nodes de exemplo
class FetchNode(Node):
    """Busca dados (sem dependencias)."""
    name = "fetch_data"
    dependencies: list[str] = []

    def run(self, ctx: PipelineContext) -> PipelineContext:
        ctx["dados"] = [10, 20, 30, 40, 50]
        return ctx


class AnalyzeNode(Node):
    """Analisa dados (depende de fetch_data)."""
    name = "analyze_data"
    dependencies = ["fetch_data"]

    def run(self, ctx: PipelineContext) -> PipelineContext:
        dados = ctx["dados"]
        ctx["media"] = sum(dados) / len(dados)
        return ctx


class ReportNode(Node):
    """Gera relatorio (depende de analyze_data)."""
    name = "report"
    dependencies = ["analyze_data"]

    def run(self, ctx: PipelineContext) -> PipelineContext:
        ctx["report"] = f"Media calculada: {ctx['media']:.1f}"
        return ctx


# Registrar e executar
engine = DAGEngine()
engine.register_many([FetchNode(), AnalyzeNode(), ReportNode()])

# Dry-run — mostra ordem sem executar
plan = engine.dry_run("report")
print("Plano de execucao:")
for step in plan:
    print(f"  -> {step}")

In [ ]:
# Executar o pipeline
ctx = engine.run("report")

print(f"Dados: {ctx['dados']}")
print(f"Media: {ctx['media']}")
print(f"Report: {ctx['report']}")

## 2. Node — ABC e Subclasses

Todo componente do pipeline e um `Node`. A interface exige:
- `name` — identificador unico no DAG
- `dependencies` — lista de names dos nodes predecessores
- `run(ctx)` — logica de execucao, recebe e retorna PipelineContext

`Node.__init_subclass__()` isola dependencias entre subclasses 
(cada subclass tem sua propria lista, nao compartilhada).

In [ ]:
# __init_subclass__ garante isolamento de dependencies
class NodeA(Node):
    name = "a"
    dependencies = ["x"]
    def run(self, ctx): return ctx

class NodeB(Node):
    name = "b"
    dependencies = ["y", "z"]
    def run(self, ctx): return ctx

# Cada classe tem suas proprias deps — nao contaminam uma a outra
print(f"NodeA.dependencies: {NodeA.dependencies}")
print(f"NodeB.dependencies: {NodeB.dependencies}")
assert NodeA.dependencies != NodeB.dependencies

## 3. PipelineContext — Comunicacao entre Nodes

`PipelineContext` e um dict tipado que carrega dados entre nodes.
Nenhum node acessa estado global — tudo via contexto.

In [ ]:
ctx = PipelineContext()

# Funciona como dict
ctx["valor"] = 42
ctx["nome"] = "teste"

print(f"valor: {ctx['valor']}")
print(f"nome: {ctx.get('nome', 'default')}")
print(f"chave inexistente: {ctx.get('outra', 'nao existe')}")

In [ ]:
# Metodo get_typed — acesso com verificacao de tipo
valor = ctx.get_typed("valor", int)
print(f"get_typed('valor', int): {valor}")

# Tracking de erros
print(f"has_errors: {ctx.has_errors}")
print(f"errors: {ctx.errors}")

# Simular erro
ctx["_errors"] = {"node_x": "ConnectionError: timeout"}
print(f"\nApos erro:")
print(f"has_errors: {ctx.has_errors}")
print(f"errors: {ctx.errors}")

## 4. Registry — Mapeamento CLI → Pipelines

O Registry mapeia nomes de pipeline (usados no CLI) para nodes terminais do DAG.
`create_engine()` registra todos os nodes do sistema.

In [ ]:
from carteira_auto.core.registry import (
    create_engine,
    get_terminal_node,
    list_pipelines,
    PIPELINE_PRESETS,
)

# Listar todos os pipelines disponiveis
pipelines = list_pipelines()
print(f"{len(pipelines)} pipelines registrados:\n")
for nome, descricao in sorted(pipelines.items()):
    terminal = PIPELINE_PRESETS[nome]
    print(f"  {nome:35s} -> {terminal:30s} | {descricao}")

In [ ]:
# Criar engine com todos os nodes
engine = create_engine()

# Resolver um pipeline
terminal = get_terminal_node("macro")
plan = engine.dry_run(terminal)

print(f"Pipeline 'macro' (terminal: {terminal}):")
for step in plan:
    print(f"  -> {step}")

In [ ]:
# Pipelines mais complexos com cadeia de dependencias
for pipeline_name in ["analyze", "risk", "rebalance"]:
    terminal = get_terminal_node(pipeline_name)
    plan = engine.dry_run(terminal)
    print(f"\n'{pipeline_name}' ({terminal}): {' -> '.join(plan)}")

## 5. Error Handling — fail_fast vs Tolerante

O DAGEngine suporta dois modos de erro:
- `fail_fast=True`: para no primeiro erro (util para debug)
- `fail_fast=False` (padrao): registra erros em `ctx["_errors"]` e continua

In [ ]:
from carteira_auto.core.engine import NodeExecutionError

class BrokenNode(Node):
    """Node que sempre falha."""
    name = "broken"
    dependencies: list[str] = []
    def run(self, ctx):
        raise ValueError("Erro intencional para demonstracao")

class SafeNode(Node):
    """Node que funciona."""
    name = "safe"
    dependencies: list[str] = []
    def run(self, ctx):
        ctx["safe_result"] = "OK"
        return ctx

# Modo tolerante (padrao) — registra erro e continua
engine_tolerante = DAGEngine(fail_fast=False)
engine_tolerante.register_many([BrokenNode(), SafeNode()])

ctx = engine_tolerante.run("safe")
print(f"safe_result: {ctx.get('safe_result')}")
print(f"has_errors: {ctx.has_errors}")
print(f"errors: {ctx.errors}")

In [ ]:
# Modo fail_fast — para no primeiro erro
engine_strict = DAGEngine(fail_fast=True)
engine_strict.register(BrokenNode())

try:
    engine_strict.run("broken")
except NodeExecutionError as e:
    print(f"NodeExecutionError capturado:")
    print(f"  Node: {e.node_name}")
    print(f"  Causa: {e.__cause__}")

## Resumo

O motor de orquestracao do carteira_auto:

1. **DAGEngine** resolve dependencias automaticamente via topological sort
2. **Node** e a unidade atomica — name + dependencies + run(ctx)
3. **PipelineContext** carrega dados entre nodes (sem estado global)
4. **Registry** mapeia nomes CLI → nodes terminais
5. **Error handling** permite execucao parcial sem interromper o pipeline

```
CLI: python -m carteira_auto run <pipeline>
  |                                        
  v                                        
Registry: get_terminal_node(pipeline) -> node_name
  |                                        
  v                                        
DAGEngine: resolve(node_name) -> [ordered nodes]
  |                                        
  v                                        
Execucao: node1.run(ctx) -> node2.run(ctx) -> ... -> resultado
```